# 1: Accessing the API

This notebook showcases all the concepts from the articles under the "Accessing Claude with the API" section.

In [ ]:
%pip install anthropic

In [ ]:
api_key = # Paste here for convenience. Use .env in real code for security
base_url = "https://api.deepseek.com/anthropic" # or "https://api.anthropic.com"
model = "deepseek-v4-flash" # or "claude-sonnet-4-0"

from anthropic import Anthropic

client = Anthropic(
    api_key=api_key,
    base_url=base_url
)

A basic request based on [Making a request](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287725)

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": "Why is a raven like a writing desk?"
        }
    ]
)

In [ ]:

print("Response text:", message.content[0].thinking) # Slightly different from tutorial
print("Full response:", message)

Helper functions based on [Multi-Turn conversations](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287735)


In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].thinking

Basic chat application based on [Chat exercise](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287727)

In [ ]:
messages = []

while True:
    user_input = input("> ")
    if user_input == "":
        break

    add_user_message(messages, user_input)
    response_message = chat(messages)
    add_assistant_message(messages, response_message)

    print("---", response_message, "---", sep="\n")

System prompt demonstration based on [System prompts](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287733) and [System prompts exercise](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287724)

In [ ]:
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].thinking

In [ ]:
# Same example used as the article
# Experiment by changing the system prompt and seeing the change in response

messages = []

add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters"
)

answer = chat(messages, "You are a Python engineer who writes very concise code")

print(answer)

Temperature demonstration based on [Temperature](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287728)

In [ ]:
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].thinking

In [ ]:
# Same example used as the article
# Experiment by changing the temperature and seeing the change in response

messages = []

add_user_message(
    messages,
    "Generate a one sentence movie idea"
)

answer = chat(messages, system="You are an experienced and helpful writer", temperature=0.4)
print(answer)

Response streaming demonstration from [Response streaming](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287734)

In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print("streamed text:", text)
        pass

    print("final message:", stream.get_final_message())

Structured data demonstration from [Structured data](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287732) and [Structured data exercise](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287729)

In [ ]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].thinking

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

print(text)

Prefiling is not supported on the Anthropic-style DeepSeek endpoint, so here is a demonstration of the concept using the OpenAI format.

In [ ]:
%pip install openai

from openai import OpenAI

client = OpenAI(
    api_key=,
    base_url="https://api.deepseek.com/beta",
)

messages = [
    {"role": "user", "content": "Generate a very short event bridge rule as json"},
    {"role": "assistant", "content": "Here is the JSON object with 3-space indentation.\n```json\n", "prefix": True}
]
response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=messages,
    stop=["```"],
)
print(response.choices[0].message.content)